# Tutorial 3 — Data Preparation via Code Objects

This notebook creates and runs Python **Code Objects** that clean and standardize
each source dataset. Each script executes on the Rhino client in an isolated
container — raw data never reaches this notebook.

**Can I do this in the UI instead?**
Yes — you can create and run Python Code Objects entirely from the FCP Dashboard.
Navigate to Dashboard → Code → New Code Object → Python Code, paste your
script, select input/output schemas, and click Run. This notebook automates this same process.

**About input/output schemas:**
This tutorial enforces input schemas (validates the source dataset matches the
expected structure before running) and uses auto-inferred output schemas
(the platform determines the output schema from the script's actual output).
Both of these are optional — you can run Code Objects with `None` for any
schema slot if you prefer no validation. Enforcing schemas is a best practice
for production pipelines but is not a requirement.

---
**Prerequisites:**
- Tutorial 1 complete — UIDs required in Configuration cell
- Login information (username & password)

**Inputs:** Registered source datasets (patients, encounters, procedures)

**Outputs:** Three cleaned "prepared" datasets registered on the FCP,
plus three auto-inferred output schemas. UIDs carried into Tutorial 4.

## Step 1: Configuration

In [ ]:
import time
import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeObjectRunInput,
    CodeTypes,
)
from rhino_health.lib.endpoints.code_run.code_run_dataclass import CodeRunStatus
from getpass import getpass
from rhino_health import ApiEnvironment

# From Tutorial 1 — paste your values here
PROJECT_UID            = "<YOUR_PROJECT_UID>"           # REPLACE with your project UID
# Dataset UIDs
PATIENTS_DATASET_UID   = "<PATIENTS_DATASET_UID>"       # REPLACE with your Patients dataset UID
ENCOUNTERS_DATASET_UID = "<ENCOUNTERS_DATASET_UID>"     # REPLACE with your Encounters dataset UID
PROCEDURES_DATASET_UID = "<PROCEDURES_DATASET_UID>"     # REPLACE with your Procedures dataset UID
# Schema UIDs
PATIENTS_SCHEMA_UID    = "<PATIENTS_SCHEMA_UID>"        # REPLACE with your Patients schema UID
ENCOUNTERS_SCHEMA_UID  = "<ENCOUNTERS_SCHEMA_UID>"      # REPLACE with your Encounters schema UID
PROCEDURES_SCHEMA_UID  = "<PROCEDURES_SCHEMA_UID>"      # REPLACE with your Procedures schema UID

# Verify that the user has filled in the required values
if (PROJECT_UID == "" or PROJECT_UID == "<YOUR_PROJECT_UID>"):
    raise ValueError("Please fill in your project UID from Tutorial 1 before running this cell.")
print(f"Using PROJECT_UID={PROJECT_UID}")

if (PATIENTS_DATASET_UID == "" or ENCOUNTERS_DATASET_UID == "" or PROCEDURES_DATASET_UID == "" or PATIENTS_DATASET_UID == "<YOUR_PATIENTS_DATASET_UID>" or ENCOUNTERS_DATASET_UID == "<YOUR_ENCOUNTERS_DATASET_UID>" or PROCEDURES_DATASET_UID == "<YOUR_PROCEDURES_DATASET_UID>"):
    raise ValueError("Please fill in the dataset UIDs from Tutorial 1 before running this cell.")
print(f"\nUsing PATIENTS_DATASET_UID={PATIENTS_DATASET_UID}")
print(f"Using ENCOUNTERS_DATASET_UID={ENCOUNTERS_DATASET_UID}")
print(f"Using PROCEDURES_DATASET_UID={PROCEDURES_DATASET_UID}")

if (PATIENTS_SCHEMA_UID == "" or ENCOUNTERS_SCHEMA_UID == "" or PROCEDURES_SCHEMA_UID == "" or PATIENTS_SCHEMA_UID == "<YOUR_PATIENTS_SCHEMA_UID>" or ENCOUNTERS_SCHEMA_UID == "<YOUR_ENCOUNTERS_SCHEMA_UID>" or PROCEDURES_SCHEMA_UID == "<YOUR_PROCEDURES_SCHEMA_UID>"):
    raise ValueError("Please fill in the schema UIDs from Tutorial 1 before running this cell.")
print(f"\nUsing PATIENTS_SCHEMA_UID={PATIENTS_SCHEMA_UID}")
print(f"Using ENCOUNTERS_SCHEMA_UID={ENCOUNTERS_SCHEMA_UID}")
print(f"Using PROCEDURES_SCHEMA_UID={PROCEDURES_SCHEMA_UID}")

## Step 2: Initialize Shared Utilities

Run this cell once. It defines helper functions used by all three dataset sections.

> Be sure to replace `my_username` with `<YOUR_USERNAME>`!

In [ ]:
def authenticate():
    """Authenticate and return a session. Called at the start of each section."""
    my_username = "<YOUR_USERNAME>"  # REPLACE
    if my_username in ("", "<YOUR_EMAIL>", "<YOUR_USERNAME>"):
        raise ValueError("Please fill in your email/username in the authenticate() function before running this cell.")
    session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL) # e.g., PROD_AWS_URL, STAGING_AWS_URL, DEV2_AWS_URL, SOLUTIONS_GCP_URL
    print(f"Logged in successfully as <{my_username}>.")
    return session


def register_or_reuse_code_object(session, name, description, code,
                                   input_schema_uids, project_uid):
    """
    Register a Python Code Object if one with this name doesn't already exist.

    - input_schema_uids: validates input dataset structure before running (recommended)
    - output_data_schema_uids=[None]: auto-infer output schema from actual output CSV
    - Pass [None] for any slot to skip schema enforcement for that slot
    """
    existing = session.code_object.get_code_object_by_name(name, project_uid=project_uid)
    if existing:
        print(f"Code Object '{name}' already exists — reusing: {existing.uid}")
        return existing

    co = session.code_object.create_code_object(CodeObjectCreateInput(
        name=name,
        description=description,
        project_uid=project_uid,
        code_type=CodeTypes.PYTHON_CODE,
        config={
            "python_version": "3.9",
            "requirements": ["pandas~=1.4.2", "numpy==1.22.*"],
            "python_code": code,
            "code_execution_mode": "snippet",
        },
        input_data_schema_uids=input_schema_uids,
        output_data_schema_uids=[None],  # auto-infer output schema
    ))
    print(f"Code Object registered: {co.uid}")
    return co


def run_and_poll(session, code_object_uid, input_dataset_uid, output_name, max_wait=600):
    """
    Execute a Code Object and wait for completion.
    Returns the completed CodeRun object with output_dataset_uids.
    """
    # run_code_object returns CodeObjectRunAsyncResponse (has code_run_uid, not uid)
    async_response = session.code_object.run_code_object(CodeObjectRunInput(
        code_object_uid=code_object_uid,
        input_dataset_uids=[[input_dataset_uid]],       # double-nested: List[List[str]]
        output_dataset_naming_templates=[output_name],
        timeout_seconds=max_wait,
    ))

    code_run_uid = async_response.code_run_uid
    print(f"Run initiated: {code_run_uid}")
    print(f"FCP UI: Dashboard → Code Runs → find this UID to monitor progress")

    # Fetch the CodeRun object (which has wait_for_completion)
    code_run = session.code_run.get_code_run(code_run_uid)
    result = code_run.wait_for_completion(timeout_seconds=max_wait, print_progress=True)

    if result.status not in (CodeRunStatus.COMPLETED, CodeRunStatus.HALTED_SUCCESS):
        raise RuntimeError(f"Code Object run ended with status: {result.status.value}")

    print(f"\nRun completed successfully.")
    return result


print("Utility functions loaded.")

---
# Step 3: Dataset Prep (Patients)

**Transformations applied:**
- Normalize `Gender` to title case — fixes `"female"`, `"FEMALE"`, `"MALE"` etc.
- Drop rows with `Gender` values not in the allowed set (`Male`, `Female`, `Other`)
  after normalization — catches truly invalid values like `"Alien"`
- Drop rows with null `Gender` — required field for OMOP Person mapping
- Validate `YearOfBirth` is between 1900–2025 — drops `10000` and similar errors
- Drop exact duplicate rows (the CSV intentionally has one)
- Drop rows missing `patientID`

**This section is fully independent** — run it without the others if needed.
The script runs entirely on the Rhino client. Only the output dataset metadata
is returned to the FCP cloud.

### Define the Patients Prep Script

The script below runs **inside an isolated container on the Rhino client**.
It reads from `/input/dataset.csv` (single input) and writes to
`/output/dataset.csv`. Do NOT use the Rhino SDK inside a Code Object script —
the SDK is only available in your orchestrating notebook, not inside containers.

In [ ]:
PATIENTS_PREP_CODE = '''
import pandas as pd
import os, json

os.makedirs("/output", exist_ok=True)
params = json.load(open("/input/run_params.json")) if os.path.exists("/input/run_params.json") else {}

# Single input → /input/dataset.csv (not /input/0/dataset.csv)
df = pd.read_csv("/input/dataset.csv")
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")

# 1. Normalize Gender to title case (Male / Female / Other)
df["Gender"] = df["Gender"].str.strip().str.title()
print(f"  Gender values after normalization: {sorted(df['Gender'].dropna().unique().tolist())}")

# 2. Drop rows with invalid Gender values (not in allowed enum)
ALLOWED_GENDER = {"Male", "Female", "Other"}
invalid_gender = df["Gender"].notna() & ~df["Gender"].isin(ALLOWED_GENDER)
n_invalid = invalid_gender.sum()
if n_invalid:
    print(f"  Dropping {n_invalid} row(s) with invalid Gender: {df.loc[invalid_gender, 'Gender'].unique().tolist()}")
df = df[~invalid_gender]

# 3. Drop rows with null Gender
n_before = len(df)
df = df[df["Gender"].notna()]
print(f"  Dropped {n_before - len(df)} row(s) with null Gender")

# 4. Validate YearOfBirth range
invalid_yob = ~df["YearOfBirth"].between(1900, 2025)
n_yob = invalid_yob.sum()
if n_yob:
    print(f"  Dropping {n_yob} row(s) with invalid YearOfBirth: {df.loc[invalid_yob, 'YearOfBirth'].unique().tolist()}")
df = df[~invalid_yob]

# 5. Drop duplicate rows
n_before = len(df)
df = df.drop_duplicates()
print(f"  Dropped {n_before - len(df)} duplicate row(s)")

# 6. Drop rows missing patientID
n_before = len(df)
df = df[df["patientID"].notna()]
if n_before - len(df):
    print(f"  Dropped {n_before - len(df)} row(s) missing patientID")

df.to_csv("/output/dataset.csv", index=False)
print(f"\\nOutput: {len(df)} rows, {len(df.columns)} columns")
print(f"Final Gender distribution: {df['Gender'].value_counts().to_dict()}")
'''

### Register and Run — Patients

In [ ]:
session = authenticate()

patients_co = register_or_reuse_code_object(
    session,
    name="Data Prep — Patients",
    description="Normalize Gender casing, drop invalid/null Gender, validate YearOfBirth, dedup, drop missing patientID.",
    code=PATIENTS_PREP_CODE,
    input_schema_uids=[PATIENTS_SCHEMA_UID],  # enforce input schema; use [None] to skip
    project_uid=PROJECT_UID,
)

In [ ]:
patients_run = run_and_poll(
    session,
    code_object_uid=patients_co.uid,
    input_dataset_uid=PATIENTS_DATASET_UID,
    output_name="Patients — Site A — Prepared",
)

prepared_patients_ds         = session.dataset.get_dataset(patients_run.output_dataset_uids.root[0].root[0].root[0])
PREPARED_PATIENTS_UID        = prepared_patients_ds.uid
PREPARED_PATIENTS_SCHEMA_UID = prepared_patients_ds.data_schema_uid
print(f"\nPrepared patients dataset: {PREPARED_PATIENTS_UID}")
print(f"Auto-inferred schema UID:  {PREPARED_PATIENTS_SCHEMA_UID}")

---
# Step 4: Dataset Prep (Encounters)

**Transformations applied:**
- Parse `DateOfService` to `YYYY-MM-DD` format — drops rows with null/unparseable dates
- Normalize `TypeOfService` to title case — fixes `"outpatient"`, `"INPATIENT"` etc.
- Drop exact duplicate rows
- Drop rows missing `patientID` or `visitID`

**This section is fully independent.**

### Define the Encounters Prep Script

In [ ]:
ENCOUNTERS_PREP_CODE = '''
import pandas as pd
import os

os.makedirs("/output", exist_ok=True)

df = pd.read_csv("/input/dataset.csv")
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")

# 1. Parse DateOfService
before_nulls = df["DateOfService"].isna().sum()
df["DateOfService"] = pd.to_datetime(df["DateOfService"], errors="coerce").dt.strftime("%Y-%m-%d")
new_nulls = df["DateOfService"].isna().sum() - before_nulls
if new_nulls:
    print(f"  {new_nulls} date(s) could not be parsed — set to null")

# Drop rows with null DateOfService (required for OMOP visit_start_date)
n_before = len(df)
df = df[df["DateOfService"].notna()]
print(f"  Dropped {n_before - len(df)} row(s) with null DateOfService")

# 2. Normalize TypeOfService to title case
df["TypeOfService"] = df["TypeOfService"].str.strip().str.title()
print(f"  TypeOfService values after normalization: {sorted(df['TypeOfService'].dropna().unique().tolist())}")

# 3. Drop duplicates
n_before = len(df)
df = df.drop_duplicates()
print(f"  Dropped {n_before - len(df)} duplicate row(s)")

# 4. Drop rows missing required IDs
for col in ["patientID", "visitID"]:
    n_before = len(df)
    df = df[df[col].notna()]
    if n_before - len(df):
        print(f"  Dropped {n_before - len(df)} row(s) missing {col}")

df.to_csv("/output/dataset.csv", index=False)
print(f"\\nOutput: {len(df)} rows")
print(f"TypeOfService distribution: {df['TypeOfService'].value_counts().to_dict()}")
'''

### Register and Run — Encounters

In [ ]:
session = authenticate()

encounters_co = register_or_reuse_code_object(
    session,
    name="Data Prep — Encounters",
    description="Parse DateOfService, normalize TypeOfService casing, dedup, drop missing IDs.",
    code=ENCOUNTERS_PREP_CODE,
    input_schema_uids=[ENCOUNTERS_SCHEMA_UID],
    project_uid=PROJECT_UID,
)

In [ ]:
encounters_run = run_and_poll(
    session,
    code_object_uid=encounters_co.uid,
    input_dataset_uid=ENCOUNTERS_DATASET_UID,
    output_name="Encounters — Site A — Prepared",
)

prepared_encounters_ds         = session.dataset.get_dataset(encounters_run.output_dataset_uids.root[0].root[0].root[0])
PREPARED_ENCOUNTERS_UID        = prepared_encounters_ds.uid
PREPARED_ENCOUNTERS_SCHEMA_UID = prepared_encounters_ds.data_schema_uid
print(f"\nPrepared encounters dataset: {PREPARED_ENCOUNTERS_UID}")

---
# Step 5: Dataset Prep (Procedures)

**Transformations applied:**
- Parse `ProcedureDate` to `YYYY-MM-DD` format
- Drop rows where `ProcedureCode` is null — cannot be mapped to OMOP without a code
- Drop exact duplicate rows
- Drop rows missing `patientID` or `visitID`

Note: null `ProcedureDescription` (row 79) is **left in place** — the description
column is not required for OMOP and its absence does not affect harmonization.

**This section is fully independent.**

### Define the Procedures Prep Script

In [ ]:
PROCEDURES_PREP_CODE = '''
import pandas as pd
import os

os.makedirs("/output", exist_ok=True)

df = pd.read_csv("/input/dataset.csv")
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")

# 1. Parse ProcedureDate
before_nulls = df["ProcedureDate"].isna().sum()
df["ProcedureDate"] = pd.to_datetime(df["ProcedureDate"], errors="coerce").dt.strftime("%Y-%m-%d")
new_nulls = df["ProcedureDate"].isna().sum() - before_nulls
if new_nulls:
    print(f"  {new_nulls} date(s) could not be parsed — set to null")

# 2. Drop rows where ProcedureCode is null
n_before = len(df)
df = df[df["ProcedureCode"].notna()]
print(f"  Dropped {n_before - len(df)} row(s) with null ProcedureCode")

# Note: null ProcedureDescription rows are kept — description is not required for OMOP
null_desc = df["ProcedureDescription"].isna().sum()
if null_desc:
    print(f"  Note: {null_desc} row(s) have null ProcedureDescription — kept (not required for OMOP)")

# 3. Drop duplicates
n_before = len(df)
df = df.drop_duplicates()
print(f"  Dropped {n_before - len(df)} duplicate row(s)")

# 4. Drop rows missing required IDs
for col in ["patientID", "visitID"]:
    n_before = len(df)
    df = df[df[col].notna()]
    if n_before - len(df):
        print(f"  Dropped {n_before - len(df)} row(s) missing {col}")

df.to_csv("/output/dataset.csv", index=False)
print(f"\\nOutput: {len(df)} rows")
print(f"ProcedureCode distribution: {df['ProcedureCode'].value_counts().to_dict()}")
'''

### Register and Run — Procedures

In [ ]:
session = authenticate()

procedures_co = register_or_reuse_code_object(
    session,
    name="Data Prep — Procedures",
    description="Parse ProcedureDate, drop null ProcedureCode rows, dedup, drop missing IDs.",
    code=PROCEDURES_PREP_CODE,
    input_schema_uids=[PROCEDURES_SCHEMA_UID],
    project_uid=PROJECT_UID,
)

In [ ]:
procedures_run = run_and_poll(
    session,
    code_object_uid=procedures_co.uid,
    input_dataset_uid=PROCEDURES_DATASET_UID,
    output_name="Procedures — Site A — Prepared",
)

prepared_procedures_ds         = session.dataset.get_dataset(procedures_run.output_dataset_uids.root[0].root[0].root[0])
PREPARED_PROCEDURES_UID        = prepared_procedures_ds.uid
PREPARED_PROCEDURES_SCHEMA_UID = prepared_procedures_ds.data_schema_uid
print(f"\nPrepared procedures dataset: {PREPARED_PROCEDURES_UID}")

## Step 6: FCP UI — What to Check After Running This Notebook

1. **Code Objects** → Dashboard → Projects → [Your Project] → **Code**
   - Three Python Code Objects appear
      * "Data Prep — Patients"
      * "Data Prep — Encounters"
      * "Data Prep — Procedures"
   - These are saved and reusable — you can run them against new data at any site
   - Select the three dots to the right of the `Run` button and select `Show code object configuration` from the dropdown menu to view setup

2. **Code Runs** → Dashboard → Projects → [Your Project] → **Code Runs**
   - Completed runs should appear under the correponding code object
   - Click any run to see: input dataset, output dataset, start/end time, status, and logs (including any `print()` output

3. **Output Datasets** → Dashboard → Projects → [Your Project] → **Datasets**
   - Three new datasets appear
      * "Patients — Site A — Prepared"
      * "Encounters — Site A — Prepared"
      * "Procedures — Site A — Prepared"
   - Row counts should be slightly less than 100 (invalid/duplicate rows removed)
   - The Schema column shows the auto-inferred output schema name
   - Click a dataset → Analytics tab → verify distributions look clean
     (e.g., Gender should now show only Male/Female/Other at consistent casing)

## Summary — Copy These UIDs

In [ ]:
print("=" * 65)
print("  Tutorial 3 Complete — save these UIDs")
print("=" * 65)
print(f"PREPARED_PATIENTS_UID          = '{PREPARED_PATIENTS_UID}'")
print(f"PREPARED_ENCOUNTERS_UID        = '{PREPARED_ENCOUNTERS_UID}'")
print(f"PREPARED_PROCEDURES_UID        = '{PREPARED_PROCEDURES_UID}'")
print(f"PREPARED_PATIENTS_SCHEMA_UID   = '{PREPARED_PATIENTS_SCHEMA_UID}'")
print(f"PREPARED_ENCOUNTERS_SCHEMA_UID = '{PREPARED_ENCOUNTERS_SCHEMA_UID}'")
print(f"PREPARED_PROCEDURES_SCHEMA_UID = '{PREPARED_PROCEDURES_SCHEMA_UID}'")
print("=" * 65)
print("\nContinue to: Tutorial 4 - Harmonization")